# 6교시. 작은 함수들을 한 줄로 연결하기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/document_ai_lecture_2026/colab/06_ocr_ai_integration.ipynb)

**목표:** 오류를 숨기지 않고 실제·mock 경로를 연결합니다.

**결과물:** `app_06.py`

- 기본 경로는 API 키와 OCR 모델 다운로드가 필요 없습니다.
- 선택 실습은 기본값이 `False`입니다.
- 실제 개인정보가 없는 합성 영수증만 사용합니다.


In [ ]:
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Output:", OUTPUT_DIR.resolve())


In [ ]:
SAMPLE_OCR_TEXT = '샘플문구점\n거래일자: 2026-07-27\n연필 2개 × 1,000원 = 2,000원\n노트 1개 × 3,000원 = 3,000원\n합계: 5,000원\n'
SAMPLE_VLM_MARKDOWN = '# 샘플문구점\n\n거래일자: 2026-07-27\n\n| 품목 | 수량 | 단가 | 금액 |\n|---|---:|---:|---:|\n| 연필 | 2 | 1,000원 | 2,000원 |\n| 노트 | 1 | 3,000원 | 3,000원 |\n\n**합계: 5,000원**\n'
SAMPLE_RECEIPT = {'document_type': 'receipt', 'store_name': '샘플문구점', 'date': '2026-07-27', 'total_amount': 5000, 'items': [{'name': '연필', 'quantity': 2, 'unit_price': 1000, 'line_total': 2000}, {'name': '노트', 'quantity': 1, 'unit_price': 3000, 'line_total': 3000}], 'source_mode': 'mock'}


## 핵심 3개

1. 단계를 작은 함수로 나눕니다.
2. 오류와 사용 모드를 화면에 표시합니다.
3. mock은 사용자가 명시적으로 선택합니다.


In [ ]:
def validate_upload(file_path):
    if not file_path:
        return ["파일을 선택하세요."]
    return []


def mock_extract(ocr_text):
    data = dict(SAMPLE_RECEIPT)
    data["source_mode"] = "mock_extraction"
    return data


def process_document(file_path=None, *, processor="ocr", use_sample=False):
    if processor not in ("ocr", "vlm"):
        return {
            "ok": False,
            "status": "입력 오류",
            "errors": ["processor는 ocr 또는 vlm이어야 합니다."],
        }

    if use_sample:
        document_text = (
            SAMPLE_VLM_MARKDOWN if processor == "vlm" else SAMPLE_OCR_TEXT
        )
        status = (
            "MOCK PaddleOCR-VL + MOCK 추출"
            if processor == "vlm"
            else "MOCK PaddleOCR + MOCK 추출"
        )
    else:
        errors = validate_upload(file_path)
        if errors:
            return {
                "ok": False,
                "status": "입력 오류",
                "errors": errors,
                "can_continue_with_sample": True,
            }
        return {
            "ok": False,
            "status": "실제 모델 선택 실행 필요",
            "errors": [
                "PaddleOCR 또는 PaddleOCR-VL 선택 실습을 실행하거나 "
                "'샘플로 계속'을 선택하세요."
            ],
            "can_continue_with_sample": True,
        }

    return {
        "ok": True,
        "status": status,
        "document_text": document_text,
        "data": mock_extract(document_text),
    }


## 실습. 오류 경로와 명시적 mock 경로 확인


In [ ]:
error_result = process_document()
assert not error_result["ok"]
assert "data" not in error_result

sample_result = process_document(processor="vlm", use_sample=True)
assert sample_result["ok"]
assert "MOCK" in sample_result["status"]
assert sample_result["data"]["total_amount"] == 5000

print(error_result["status"], "→ 사용자가 샘플 선택")
print(sample_result["status"], "→ 완료")


In [ ]:
app_code = '''from src.pipeline import process_document

def run_uploaded(file_path, processor="ocr"):
    return process_document(file_path, processor=processor)

def run_sample(processor="ocr"):
    return process_document(processor=processor, use_sample=True)

# Gradio에서는 두 함수를 서로 다른 버튼에 연결합니다.
'''
output_path = OUTPUT_DIR / "app_06.py"
output_path.write_text(app_code, encoding="utf-8")
print("저장 완료:", output_path)


## mock 대체 경로

필수 경로 자체가 명시적인 mock 실습입니다. 실제 모델 오류 뒤에 자동 전환하지 않고
오류를 확인한 뒤 처리기를 골라 `process_document(use_sample=True)`를 실행합니다.


## 확인

- 오류 결과에 관련 없는 JSON이 없는가?
- mock 상태가 화면에 분명히 표시되는가?
- 사용자가 직접 샘플 경로를 선택했는가?
